## Sankey Plot


In [1]:
import plotly.io as pio
import plotly.graph_objects as go
from collections import OrderedDict
import os
import pandas as pd
import message_ix
from ixmp import Platform
import pyam
import yaml
from yaml import SafeLoader
import numpy as np
from matplotlib import pyplot as plt
from itertools import product
from datetime import datetime
import warnings

In [2]:
# %% 7.5) Sankey diagrams (for one scenario at a time)
# Load the file
path_reporting = r"D:/COMMITTED/Models/NEST-Pakistan/output/reporting_outputs"
os.chdir(path_reporting)
scen = "COMMITTED__baseline__v"  # "T2.4_H2EH_comb_18" or "T1.3_Npi2025_NZEU"
file_reporting = "MESSAGEix-Pakistan 1_CurPol_2025-09-17--15-32.xlsx"
df = pd.read_excel(path_reporting + "/" + file_reporting, sheet_name="data")
year = 2025
hydrogen_td = 0.15
reg = 'Pakistan'
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df = pyam.IamDataFrame(df)
df = df.filter(region = 'Pakistan',keep =False)
df.aggregate_region(variable=df.variable, region="Pakistan", subregions=df.region, append=True)

In [3]:

variables_sankey = [
    "Primary Energy|Biomass",
    "Primary Energy|Coal",
    "Primary Energy|Gas",
    "Primary Energy|Geothermal",
    "Primary Energy|Hydro",
    "Primary Energy|Nuclear",
    "Primary Energy|Oil",
    "Primary Energy|Other",
    # "Primary Energy|Secondary Energy Trade",
    "Primary Energy|Solar",
    "Primary Energy|Wind",
    "Secondary Energy|Electricity|Biomass",
    "Secondary Energy|Electricity|Coal",
    "Secondary Energy|Electricity|Gas",
    "Secondary Energy|Electricity|Geothermal",
    "Secondary Energy|Electricity|Hydro",
    "Secondary Energy|Electricity|Nuclear",
    "Secondary Energy|Electricity|Oil",
    "Secondary Energy|Electricity|Other",
    "Secondary Energy|Electricity|Solar",
    "Secondary Energy|Electricity|Wind",
    "Secondary Energy|Gases|Biomass",
    "Secondary Energy|Gases|Coal",
    "Secondary Energy|Gases|Natural Gas",
   # "Secondary Energy|Gases|Other",   # No need because it is already taken into account in the final level
    "Secondary Energy|Heat|Biomass",
    "Secondary Energy|Heat|Coal",
    "Secondary Energy|Heat|Gas",
    "Secondary Energy|Heat|Geothermal",
    "Secondary Energy|Heat|Oil",
    "Secondary Energy|Heat|Other",
    "Secondary Energy|Hydrogen|Biomass",
    "Secondary Energy|Hydrogen|Coal",
    "Secondary Energy|Hydrogen|Electricity",
    "Secondary Energy|Hydrogen|Gas",
    "Secondary Energy|Liquids|Biomass",
    "Secondary Energy|Liquids|Coal",
    "Secondary Energy|Liquids|Gas",
    "Secondary Energy|Liquids|Oil",
    # "Secondary Energy|Solids|Biomass",
    # "Secondary Energy|Solids|Coal",
    "Final Energy|Industry|Electricity",
    "Final Energy|Industry|Gases",
    "Final Energy|Industry|Heat",
    "Final Energy|Industry|Hydrogen",
    "Final Energy|Industry|Liquids",
    # "Final Energy|Industry|Other",
    "Final Energy|Industry|Solids|Biomass",
    "Final Energy|Industry|Solids|Coal"
    "Final Energy|Non-Energy Use|Biomass",
    "Final Energy|Non-Energy Use|Coal",
    "Final Energy|Non-Energy Use|Gas",
    "Final Energy|Non-Energy Use|Oil",
    "Final Energy|Residential and Commercial|Electricity",
    "Final Energy|Residential and Commercial|Gases",
    "Final Energy|Residential and Commercial|Heat",
    "Final Energy|Residential and Commercial|Hydrogen",
    "Final Energy|Residential and Commercial|Liquids",
    # "Final Energy|Residential and Commercial|Other",
    "Final Energy|Residential and Commercial|Solids|Biomass",
    "Final Energy|Residential and Commercial|Solids|Coal",
    "Final Energy|Transportation|Electricity",
    "Final Energy|Transportation|Gases",
    "Final Energy|Transportation|Hydrogen",
    "Final Energy|Transportation|Liquids",
    "Final Energy|Transportation|Other",
    "Trade|Primary Energy|Biomass|Volume",
    "Trade|Primary Energy|Coal|Volume",
    "Trade|Primary Energy|Gas|Volume",
    "Trade|Primary Energy|Oil|Volume",
    "Trade|Secondary Energy|Electricity|Volume",
    "Trade|Secondary Energy|Hydrogen|Volume",
    "Trade|Secondary Energy|Liquids|Biomass|Volume",
    "Trade|Secondary Energy|Liquids|Coal|Volume",
    "Trade|Secondary Energy|Liquids|Oil|Volume",
]

df = df.filter(year=year, variable=variables_sankey, region=reg)

# Dump the data in Excel
d = df.timeseries().reset_index().groupby("variable").sum(numeric_only=True)
d.to_excel(path_reporting + "results_sankey.xlsx")

In [4]:

# %% Sankey Diagrams
import plotly.io as pio
import plotly.graph_objects as go
from collections import OrderedDict

trade_show = False
pio.renderers.default = 'browser'

# Load Data
df = d.copy().reset_index()
df.columns = df.columns.map(str)
year = df.columns[1]

# Step 1: Rename variables
df['variable'] = df['variable'].replace({
    'Primary Energy|Gas': 'Primary Energy|Natural Gas',
    'Secondary Energy|Electricity|Gas': 'Secondary Energy|Electricity|Gases',
    'Secondary Energy|Electricity|Oil': 'Secondary Energy|Electricity|Liquids',
    'Secondary Energy|Heat|Other': 'Secondary Energy|Heat|Electricity',
    'Secondary Energy|Hydrogen|Gas': 'Secondary Energy|Hydrogen|Natural Gas',
    'Secondary Energy|Liquids|Gas': 'Secondary Energy|Liquids|Natural Gas',
    "Final Energy|Industry|Solids|Biomass": "Final Energy|Industry|Biomass",
    "Final Energy|Industry|Solids|Coal": "Final Energy|Industry|Coal",
    'Final Energy|Non-Energy Use|Gas': 'Final Energy|Non-Energy Use|Gases',
    "Final Energy|Residential and Commercial|Solids|Biomass": "Final Energy|Residential and Commercial|Biomass",
    "Final Energy|Residential and Commercial|Solids|Coal": "Final Energy|Residential and Commercial|Coal",
    "Trade|Primary Energy|Gas|Volume": "Trade|Primary Energy|Natural Gas|Volume",
})

# Parse structure
df['parts'] = df['variable'].str.split('|')
df['level'] = df['parts'].str[0]

# Segment levels
primary_df = df[df['level'] == 'Primary Energy'].copy()
secondary_df = df[df['level'] == 'Secondary Energy'].copy()
final_df = df[df['level'] == 'Final Energy'].copy()

# Map sectors
sector_map = {
    "Industry": "Industry",
    "Transportation": "Transportation",
    "Non-Energy Use": "Non-energy",
    "Residential and Commercial": "Buildings"
}
final_df['sector'] = final_df['parts'].str[1].map(sector_map)
final_df['energy_type'] = final_df['parts'].str[2]

# Extract secondary energy links
secondary_df['output_type'] = secondary_df['parts'].str[1]
secondary_df['input_source'] = secondary_df['parts'].str[2]

# --- CREATE ORDERED NODE LIST ---
nodes = []
nodes.append("Imports")
nodes += list(primary_df['parts'].str[1].unique())
nodes += list(secondary_df['input_source'].unique())
nodes += list(secondary_df['output_type'].unique())
nodes += list(final_df['energy_type'].unique())
nodes += list(final_df['sector'].unique())
nodes += ["Losses", "Exports"]
nodes = list(OrderedDict.fromkeys(nodes))
node_index = {name: i for i, name in enumerate(nodes)}

# --- CREATE LINKS ---
primary_secondary_links = secondary_df.copy()
primary_secondary_links['source'] = primary_secondary_links['input_source'].map(node_index)
primary_secondary_links['target'] = primary_secondary_links['output_type'].map(node_index)
primary_secondary_links['value'] = primary_secondary_links[year]

secondary_final_links = final_df.copy()
secondary_final_links['source'] = secondary_final_links['energy_type'].map(node_index)
secondary_final_links['target'] = secondary_final_links['sector'].map(node_index)
secondary_final_links['value'] = secondary_final_links[year]

# --- CALCULATE LOSSES ---
secondary_inputs = secondary_df.groupby('output_type')[year].sum()
outputs_from_secondary = pd.Series(0.0, index=secondary_inputs.index)
final_outputs_by_type = final_df.groupby('energy_type')[year].sum()
for etype in final_outputs_by_type.index:
    if etype in outputs_from_secondary:
        outputs_from_secondary[etype] += final_outputs_by_type[etype]
for row in secondary_df.itertuples():
    if row.input_source in outputs_from_secondary:
        outputs_from_secondary[row.input_source] += getattr(row, "_2")
losses = []
for etype in ['Electricity', 'Gases', 'Liquids']:
    produced = secondary_inputs.get(etype, 0)
    delivered = outputs_from_secondary.get(etype, 0)
    if produced > delivered:
        losses.append({
            'source': node_index[etype],
            'target': node_index['Losses'],
            'value': produced - delivered,
            'label': f"{etype} losses"
        })

# Biomass losses
primary_biomass = primary_df[primary_df['parts'].str[1] == 'Biomass'][year].sum()
secondary_biomass = secondary_df[secondary_df['input_source'] == 'Biomass'][year].sum()
final_biomass = final_df[final_df['energy_type'] == 'Biomass'][year].sum()
biomass_loss = primary_biomass - (secondary_biomass + final_biomass)
if biomass_loss > 0:
    losses.append({
        'source': node_index['Biomass'],
        'target': node_index['Losses'],
        'value': biomass_loss,
        'label': "Biomass losses"
    })

# --- REBUILD Secondary → Final Links (Hydrogen adjusted) ---
secondary_final_links = final_df.copy()
secondary_final_links['source'] = secondary_final_links['energy_type'].map(node_index)
secondary_final_links['target'] = secondary_final_links['sector'].map(node_index)
secondary_final_links['value'] = secondary_final_links[year]

trade_links = []
if trade_show:
    # --- HANDLE TRADE FLOWS ---
    trade_df = df[df['variable'].str.startswith('Trade|')].copy()
    trade_df['parts'] = trade_df['variable'].str.split('|')
    for row in trade_df.itertuples():
        level = row.parts[1]
        fuel = row.parts[2]
        if len(row.parts) > 4:
            subfuel = row.parts[3]
            target_fuel = fuel
        else:
            target_fuel = fuel
        trade_value = getattr(row, "_2")
        if target_fuel not in node_index:
            continue
        if trade_value < 0:
            trade_links.append({
                'source': node_index['Imports'],
                'target': node_index[target_fuel],
                'value': abs(trade_value)
            })
        elif trade_value > 0:
            trade_links.append({
                'source': node_index[target_fuel],
                'target': node_index['Exports'],
                'value': trade_value
            })

# Hydrogen conversion loss (15%)
hydrogen_produced = secondary_inputs.get('Hydrogen', 0)
if trade_show:
    hydrogen_produced =- trade_df.loc[
        trade_df["variable"] == "Trade|Secondary Energy|Hydrogen|Volume", f'{year}'].sum()

hydrogen_loss = hydrogen_td * hydrogen_produced
if hydrogen_produced > 0:
    final_df.loc[final_df['energy_type'] == 'Hydrogen', year] *= (1 - hydrogen_td)
    # final_hydrogen = final_df[final_df['energy_type'] == 'Hydrogen'][year].sum()
    # Hydrogen_loss = hydrogen_produced - final_hydrogen
    losses.append({
        'source': node_index['Hydrogen'],
        'target': node_index['Losses'],
        'value': hydrogen_loss,
        'label': "Hydrogen losses"
    })


# Combine all links
all_links = pd.concat([
    primary_secondary_links[['source', 'target', 'value']],
    secondary_final_links[['source', 'target', 'value']],
    pd.DataFrame(losses),
    pd.DataFrame(trade_links)
], ignore_index=True)

# --- VISUALIZATION ---
carrier_colors = {
    'Electricity': 'salmon',
    'Gases': 'cyan',
    'Liquids': 'rosybrown',
    'Hydrogen': 'purple',
    'Heat': 'slateblue',
    'Solids': 'gray',
    'Natural Gas': 'darkcyan',
    'Biomass': 'olive',
    'Coal': 'darkgrey',
    'Losses': 'red',
    'Imports': 'green',
    'Exports': 'orange',
    "Solar": "orange",
    "Nuclear": "indianred",
    "Wind": "lightblue",
    "Hydro": "blue",
    "Oil": "black",
}

link_colors = []
for src_idx in all_links['source']:
    src_name = nodes[src_idx]
    matched = next((carrier_colors[k] for k in carrier_colors if k in src_name), 'lightgray')
    link_colors.append(matched)

# Set x and y positions aligned with node_index
x_positions = []
y_positions = []
if trade_show:
    for label in nodes:
        if label == "Imports":
            x = 0
        elif label in primary_df['parts'].str[1].unique():
            x = 0.2
        else:
            x = 0
        x_positions.append(x)

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=nodes,
        x=x_positions,
        y=y_positions
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors,
        customdata=all_links['value'],
        hovertemplate='Flow: %{value:.2f} PJ<br>Source: %{source.label}<br>Target: %{target.label}<extra></extra>'
    )
)])

# --- Annotations ---
total_primary = primary_df[year].sum()
total_final = final_df[year].sum()
total_losses = sum([l['value'] for l in losses])
imports_total = sum(link['value'] for link in trade_links if link['source'] == node_index['Imports'])
exports_total = sum(link['value'] for link in trade_links if link['target'] == node_index['Exports'])
net_trade = imports_total - exports_total

fig.add_annotation(x=0.01, y=1.05, xref="paper", yref="paper", text=f"Total Primary Energy (including Imports): {total_primary:.2f}", showarrow=False, font=dict(size=14))
fig.add_annotation(x=0.45, y=1.05, xref="paper", yref="paper", text=f"Total Losses: {total_losses:.2f}", showarrow=False, font=dict(size=14, color="red"))
fig.add_annotation(x=0.85, y=1.05, xref="paper", yref="paper", text=f"Total Final Energy: {total_final:.2f}", showarrow=False, font=dict(size=14))
fig.add_annotation(x=0.25, y=1.05, xref="paper", yref="paper", text=f"Net Trade: {net_trade:.2f} PJ", showarrow=False, font=dict(size=14, color="blue"))

if not trade_show:
    tag = " (without trade flows)"
    fname = f"sankey_diagram_{scen}_{year}.html"
else:
    tag = ""
    fname = f"sankey_diagram_{scen}_{year}_trade.html"
fig.update_layout(title_text=f"Energy Balances in {year} under {scen} scenario {tag}",
                  font_size=10,
                  # width=1200, height=600,
                  )
fig.write_html(fname)
fig.show()

In [5]:
year

'2025'